In [73]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import sqlite3
import warnings
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_squared_error, r2_score,
    accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.2f}'.format)

print('All libraries loaded successfully.')

All libraries loaded successfully.


In [74]:
# ── Load raw CSVs ──────────────────────────────────────────────────────────────
gk = pd.read_csv('/Users/zhuoyutan/Desktop/uchicago/data analysis/final project/all_goalkeepers.csv')
matches = pd.read_csv('/Users/zhuoyutan/Desktop/uchicago/data analysis/final project/matches.csv', low_memory=False)

# Strip BOM from matches column name if present
matches.columns = matches.columns.str.lstrip('\ufeff')

# Trim whitespace from Club abbreviations
gk['Club'] = gk['Club'].str.strip()

print(f'Goalkeepers: {gk.shape[0]} rows × {gk.shape[1]} cols')
print(f'Matches:     {matches.shape[0]} rows × {matches.shape[1]} cols')

gk.head(3)

# ── Filter both datasets to 2012 and later ──────────────────────────────────
gk      = gk[gk['Year'] >= 2012].reset_index(drop=True)
matches = matches[matches['year'] >= 2012].reset_index(drop=True)
print(f'After 2012 filter — Goalkeepers: {gk.shape[0]} rows | Matches: {matches.shape[0]} rows')


Goalkeepers: 2006 rows × 19 cols
Matches:     7289 rows × 209 cols
After 2012 filter — Goalkeepers: 989 rows | Matches: 4025 rows


In [75]:
# ── Build a club abbreviation → full-name mapping ──────────────────────────────
club_map = {
    'DAL': 'Dallas',
    'MET': 'MetroStars',
    'TB':  'Tampa Bay',
    'LA':  'LA Galaxy',
    'KC':  'KC Wiz',
    'COL': 'Colorado',
    'SJ':  'San Jose',
    'NE':  'New England',
    'DC':  '.',
    'CLB': 'Columbus',
    'CHI': 'Chicago',
    'MIA': 'Miami',
    'RSL': 'Real Salt Lake',
    'ATL': 'Atlanta United',
    'CHV': 'Chivas USA',
    'HOU': 'Houston',
    'NY':  'New York Red Bulls',
    'TOR': 'Toronto FC',
    'SEA': 'Seattle Sounders',
    'NYC': 'NYCFC',
    'POR': 'Portland',
    'VAN': 'Vancouver',
    'MTL': 'Montreal',
    'SKC': 'Sporting KC',
    'ORL': 'Orlando City',
    'NYRB':'New York Red Bulls',
    'MN': 'Minnesota United',
    'LAFC':'LAFC',
    'CIN': 'FC Cincinnati',
    'NAS': 'Nashville SC',
    'IFC': 'Inter Miami',
}

gk['Club_Full'] = gk['Club'].map(club_map).fillna(gk['Club'])
print('Mapping applied. Sample:')
gk[['Club','Club_Full','Year']].drop_duplicates().head(8)

Mapping applied. Sample:


,Club,Club_Full,Year
0,KC,KC Wiz,2012
1,MTL,Montreal,2012
2,CLB,Columbus,2012
3,HOU,Houston,2012
4,SJ,San Jose,2012
5,COL,Colorado,2012
6,CHV,Chivas USA,2012
7,RSL,Real Salt Lake,2012


In [76]:
# ── Basic data-type cleanup ────────────────────────────────────────────────────
# Convert attendance in matches to numeric (has commas)
matches['attendance_num'] = (
    matches['attendance'].astype(str)
    .str.replace(',', '', regex=False)
    .pipe(pd.to_numeric, errors='coerce')
)

# Ensure numeric score columns
matches['home_score'] = pd.to_numeric(matches['home_score'], errors='coerce')
matches['away_score'] = pd.to_numeric(matches['away_score'], errors='coerce')

# Total goals per match
matches['total_goals'] = matches['home_score'] + matches['away_score']

# Win-loss label for GK (1 = winning record, 0 = not)
gk['winning_record'] = (gk['W'] > gk['L']).astype(int)

print('Dtypes cleaned. GK winning_record distribution:')
print(gk['winning_record'].value_counts())

Dtypes cleaned. GK winning_record distribution:
winning_record
0    800
1    189
Name: count, dtype: int64


In [77]:
# ── Missing Value Treatment ──────────────────────────────────────────────────
# Drop rows in goalkeepers where Club or Club_Full is null
# These are players with no club assignment — not useful for analysis
print(f'GK rows before drop: {gk.shape[0]}')
gk = gk.dropna(subset=['Club', 'Club_Full']).reset_index(drop=True)
print(f'GK rows after dropping null Club/Club_Full: {gk.shape[0]}')

# Drop the referee column entirely — 100% missing after 2012 filter
if 'referee' in matches.columns:
   matches = matches.drop(columns=['referee'])
   print('Dropped: referee column (100% missing)')
print(f'Matches rows before processing: {matches.shape[0]}')

# Fill missing attendance with median

attendance_median = matches['attendance_num'].median()

matches['attendance_num'] = matches['attendance_num'].fillna(attendance_median)

print(f'Median attendance used: {attendance_median}')
print(matches['attendance_num'].isnull().sum())

print(f'Matches rows after processing: {matches.shape[0]}')
print(f'Median attendance used: {attendance_median}')

GK rows before drop: 989
GK rows after dropping null Club/Club_Full: 650
Dropped: referee column (100% missing)
Matches rows before processing: 4025
Median attendance used: 19019.5
0
Matches rows after processing: 4025
Median attendance used: 19019.5


In [78]:
# ── Load DataFrames into SQLite ────────────────────────────────────────────────
conn = sqlite3.connect(':memory:')
gk.to_sql('goalkeepers', conn, index=False, if_exists='replace')
matches.to_sql('matches', conn, index=False, if_exists='replace')
print('Tables loaded into SQLite:', [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()])

Tables loaded into SQLite: ['goalkeepers', 'matches']


In [79]:
# ─────────────────────────────────────────────────────────────────────────────
# Query 1 — Basic overview: count rows and seasons in each table.
# WHY: Verify data loaded correctly and understand temporal coverage.
# HOW: Simple COUNT + MIN/MAX aggregation on each table.
# ─────────────────────────────────────────────────────────────────────────────
q1 = pd.read_sql_query("""
    SELECT
        'goalkeepers' AS table_name,
        COUNT(*)        AS total_rows,
        MIN(Year)       AS first_year,
        MAX(Year)       AS last_year,
        COUNT(DISTINCT Club) AS unique_clubs
    FROM goalkeepers
    UNION ALL
    SELECT
        'matches',
        COUNT(*),
        MIN(year),
        MAX(year),
        COUNT(DISTINCT home)
    FROM matches
""", conn)
print('Query 1 — Dataset Overview')
q1

Query 1 — Dataset Overview


,table_name,total_rows,first_year,last_year,unique_clubs
0,goalkeepers,650,2012,2020,34
1,matches,4025,2012,2022,33


In [80]:
# ─────────────────────────────────────────────────────────────────────────────
# Query 2 — GROUP BY: Average goalkeeper metrics by year (regular season only).
# WHY: Identify whether defensive play has improved over MLS history.
# HOW: GROUP BY Year, compute AVG on key performance indicators.
# ─────────────────────────────────────────────────────────────────────────────
q2 = pd.read_sql_query("""
    SELECT
        Year,
        COUNT(*)              AS num_keepers,
        ROUND(AVG(GAA), 3)    AS avg_GAA,
        ROUND(AVG("Sv%"), 2)  AS avg_save_pct,
        ROUND(AVG(ShO), 2)    AS avg_shutouts,
        ROUND(AVG("W%"), 2)   AS avg_win_pct
    FROM goalkeepers
    WHERE Season = 'reg'
    GROUP BY Year
    ORDER BY Year
""", conn)
print('Query 2 — Avg GK Metrics by Year (regular season)')
q2.head(10)

Query 2 — Avg GK Metrics by Year (regular season)


,Year,num_keepers,avg_GAA,avg_save_pct,avg_shutouts,avg_win_pct
0,2012,62,0.93,44.25,2.68,20.01
1,2013,64,0.91,42.19,2.78,21.14
2,2014,63,0.94,38.31,2.48,18.77
3,2015,66,0.95,43.56,2.68,22.41
4,2016,66,1.02,42.70,2.53,19.68
5,2017,75,1.21,39.93,2.52,21.17
6,2018,79,1.06,39.77,2.11,20.84
7,2019,82,0.98,38.76,2.26,21.47
8,2020,93,1.00,40.36,1.59,20.88


In [81]:
# ─────────────────────────────────────────────────────────────────────────────
# Query 3 — GROUP BY: Goals allowed and saves per club across all years.
# WHY: Spot which franchises have historically been strong or weak defensively.
# HOW: GROUP BY Club, SUM goals-allowed and saves, compute overall save%.
# ─────────────────────────────────────────────────────────────────────────────
q3 = pd.read_sql_query("""
    SELECT
        Club,
        COUNT(DISTINCT Year)              AS seasons,
        SUM(GA)                           AS total_goals_allowed,
        SUM(SV)                           AS total_saves,
        ROUND(SUM(SV) * 100.0 /
              NULLIF(SUM(SV) + SUM(GA), 0), 2) AS career_save_pct,
        SUM(ShO)                          AS total_shutouts
    FROM goalkeepers
    WHERE Season = 'reg'
    GROUP BY Club
    ORDER BY career_save_pct DESC
""", conn)
print('Query 3 — Career Defensive Stats by Club')
q3

Query 3 — Career Defensive Stats by Club


,Club,seasons,total_goals_allowed,total_saves,career_save_pct,total_shutouts
0,SEA,9,339,865,71.84,82
1,MIN,5,165,407,71.15,37
2,DC,9,472,1144,70.79,88
3,NE,9,228,541,70.35,38
4,KC,5,108,253,70.08,42
5,RBNY,9,63,145,69.71,8
6,POR,9,575,1307,69.45,97
7,SKC,9,278,625,69.21,55
8,ORL,9,216,484,69.14,33
9,DAL,9,302,667,68.83,64


In [82]:
# ─────────────────────────────────────────────────────────────────────────────
# Query 4 — JOIN: Attach goalkeeper save% to matches where they were
#           the home or away keeper's club in the same year.
# WHY: Link match-level outcomes to the keeper's season performance.
# HOW: INNER JOIN goalkeepers to matches on Club_Full = home AND Year = year;
#      repeat for away side, then UNION ALL.
# ─────────────────────────────────────────────────────────────────────────────
q4 = pd.read_sql_query("""
    SELECT
        g.Player,
        g.Club,
        g.Year,
        ROUND(g."Sv%", 2)   AS save_pct,
        m.home,
        m.away,
        m.home_score,
        m.away_score,
        'home' AS side
    FROM goalkeepers g
    INNER JOIN matches m
        ON g.Club_Full = m.home
        AND g.Year    = m.year

    UNION ALL

    SELECT
        g.Player,
        g.Club,
        g.Year,
        ROUND(g."Sv%", 2),
        m.home,
        m.away,
        m.home_score,
        m.away_score,
        'away'
    FROM goalkeepers g
    INNER JOIN matches m
        ON g.Club_Full = m.away
        AND g.Year    = m.year
    LIMIT 100
""", conn)
print(f'Query 4 — GK–Match JOIN ({len(q4)} preview rows)')
q4.head(8)

Query 4 — GK–Match JOIN (100 preview rows)


,Player,Club,Year,save_pct,home,away,home_score,away_score,side
0,Dan Kennedy,CHV,2012,65.30,Chivas USA,Chicago Fire FC,1,2,home
1,Dan Kennedy,CHV,2012,65.30,Chivas USA,Colorado Rapids,0,2,home
2,Dan Kennedy,CHV,2012,65.30,Chivas USA,FC Dallas,1,1,home
3,Dan Kennedy,CHV,2012,65.30,Chivas USA,Houston Dynamo,0,1,home
4,Dan Kennedy,CHV,2012,65.30,Chivas USA,LA Galaxy,0,4,home
5,Dan Kennedy,CHV,2012,65.30,Chivas USA,LA Galaxy,1,0,home
6,Dan Kennedy,CHV,2012,65.30,Chivas USA,Montreal Impact,2,1,home
7,Dan Kennedy,CHV,2012,65.30,Chivas USA,Philadelphia Union,0,1,home


In [83]:
# ─────────────────────────────────────────────────────────────────────────────
# Query 5 — JOIN: Average attendance per year joined with avg goals per match.
# WHY: Understand whether higher-scoring games drive attendance.
# HOW: Sub-aggregate matches by year, JOIN to goalkeeper year summary.
# ─────────────────────────────────────────────────────────────────────────────
q5 = pd.read_sql_query("""
    SELECT
        m.year,
        ROUND(AVG(m.attendance_num), 0)       AS avg_attendance,
        ROUND(AVG(m.home_score + m.away_score), 2) AS avg_goals_per_match,
        ROUND(g.avg_save_pct, 2)              AS avg_keeper_save_pct
    FROM matches m
    LEFT JOIN (
        SELECT Year, AVG("Sv%") AS avg_save_pct
        FROM goalkeepers
        WHERE Season = 'reg'
        GROUP BY Year
    ) g ON m.year = g.Year
    WHERE m.home_score IS NOT NULL
    GROUP BY m.year
    ORDER BY m.year
""", conn)
print('Query 5 — Attendance × Goals × Save% by Year (JOIN)')
q5.head(10)

Query 5 — Attendance × Goals × Save% by Year (JOIN)


,year,avg_attendance,avg_goals_per_match,avg_keeper_save_pct
0,2012,18965.00,2.63,44.25
1,2013,18786.00,2.62,42.19
2,2014,19349.00,2.86,38.31
3,2015,21622.00,2.75,43.56
4,2016,21982.00,2.82,42.70
5,2017,22311.00,2.93,39.93
6,2018,22174.00,3.18,39.77
7,2019,21703.00,3.07,38.76
8,2020,17461.00,2.88,40.36
9,2021,17574.00,2.78,NaN


In [84]:
# ─────────────────────────────────────────────────────────────────────────────
# Query 6 — JOIN + aggregate: Top home vs away win rates by club.
# WHY: Quantify home-field advantage in MLS history.
# HOW: JOIN home & away results for each club, compute win rates.
# ─────────────────────────────────────────────────────────────────────────────
q6 = pd.read_sql_query("""
    SELECT
        club,
        SUM(home_games)  AS home_games,
        SUM(home_wins)   AS home_wins,
        ROUND(100.0 * SUM(home_wins) / NULLIF(SUM(home_games), 0), 1) AS home_win_pct,
        SUM(away_games)  AS away_games,
        SUM(away_wins)   AS away_wins,
        ROUND(100.0 * SUM(away_wins) / NULLIF(SUM(away_games), 0), 1) AS away_win_pct
    FROM (
        -- Home side
        SELECT
            home AS club,
            COUNT(*) AS home_games,
            SUM(CASE WHEN home_score > away_score THEN 1 ELSE 0 END) AS home_wins,
            0 AS away_games, 0 AS away_wins
        FROM matches
        WHERE home_score IS NOT NULL
        GROUP BY home

        UNION ALL

        -- Away side
        SELECT
            away AS club,
            0, 0,
            COUNT(*),
            SUM(CASE WHEN away_score > home_score THEN 1 ELSE 0 END)
        FROM matches
        WHERE away_score IS NOT NULL
        GROUP BY away
    )
    GROUP BY club
    HAVING SUM(home_games) >= 20
    ORDER BY home_win_pct DESC
    LIMIT 15
""", conn)
print('Query 6 — Home vs Away Win % by Club')
q6

Query 6 — Home vs Away Win % by Club


,club,home_games,home_wins,home_win_pct,away_games,away_wins,away_win_pct
0,Seattle Sounders FC,197,124,62.90,188,60,31.90
1,New York Red Bulls,183,111,60.70,191,59,30.90
2,LAFC,79,47,59.50,70,25,35.70
3,Atlanta United FC,98,57,58.20,90,28,31.10
4,New York City FC,131,74,56.50,130,42,32.30
5,Columbus Crew SC,159,89,56.00,156,35,22.40
6,Real Salt Lake,181,101,55.80,186,46,24.70
7,CF Montréal,27,15,55.60,27,7,25.90
8,FC Dallas,182,100,54.90,180,37,20.60
9,LA Galaxy,186,101,54.30,181,48,26.50


In [85]:
# ─────────────────────────────────────────────────────────────────────────────
# Query 7 — WINDOW FUNCTION: Rank goalkeepers by Save% within each year.
# WHY: Identify the best keeper per season without losing other rows.
# HOW: RANK() OVER (PARTITION BY Year ORDER BY Sv% DESC) — window function.
# ─────────────────────────────────────────────────────────────────────────────
q7 = pd.read_sql_query("""
    SELECT
        Year,
        Player,
        Club,
        ROUND("Sv%", 2)   AS save_pct,
        GP,
        GA,
        ShO,
        RANK() OVER (
            PARTITION BY Year
            ORDER BY "Sv%" DESC
        ) AS season_rank
    FROM goalkeepers
    WHERE Season = 'reg'
      AND GP >= 10
    ORDER BY Year, season_rank
""", conn)

# Show only the #1 keeper each year
best_per_year = q7[q7['season_rank'] == 1].reset_index(drop=True)
print('Query 7 — Best Keeper (by Save%) Each Season (min 10 GP)')
best_per_year

Query 7 — Best Keeper (by Save%) Each Season (min 10 GP)


,Year,Player,Club,save_pct,GP,GA,ShO,season_rank
0,2012,Bill Hamid,DC,77.90,24,24,8,1
1,2013,Matt Reis,NE,78.40,12,8,5,1
2,2014,Andy Gruenebaum,KC,75.60,11,12,3,1
3,2015,Jesse Gonzalez,DAL,78.70,11,10,5,1
4,2016,Tyler Deric,HOU,83.80,10,6,4,1
5,2017,Brad Guzan,ATL,80.90,14,10,8,1
6,2018,Bill Hamid,DC,81.00,14,12,5,1
7,2019,Steve Clark,POR,77.10,24,25,6,1
8,2020,Jimmy Maurer,DAL,79.40,16,13,7,1


In [86]:
# ─────────────────────────────────────────────────────────────────────────────
# Query 8 — WINDOW FUNCTION: Running total of league goals scored by year.
# WHY: Show cumulative offensive output growth as MLS expanded.
# HOW: SUM() OVER (ORDER BY year ROWS UNBOUNDED PRECEDING) — running sum.
# ─────────────────────────────────────────────────────────────────────────────
q8 = pd.read_sql_query("""
    WITH yearly AS (
        SELECT
            year,
            COUNT(*)                           AS num_matches,
            SUM(home_score + away_score)        AS total_goals
        FROM matches
        WHERE home_score IS NOT NULL
        GROUP BY year
    )
    SELECT
        year,
        num_matches,
        total_goals,
        ROUND(CAST(total_goals AS REAL) / num_matches, 2) AS goals_per_match,
        SUM(total_goals) OVER (
            ORDER BY year
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cumulative_goals
    FROM yearly
    ORDER BY year
""", conn)
print('Query 8 — Running Total of League Goals by Year (Window Function)')
q8

Query 8 — Running Total of League Goals by Year (Window Function)


,year,num_matches,total_goals,goals_per_match,cumulative_goals
0,2012,338,888,2.63,888
1,2013,338,887,2.62,1775
2,2014,338,966,2.86,2741
3,2015,357,983,2.75,3724
4,2016,357,1006,2.82,4730
5,2017,391,1144,2.93,5874
6,2018,408,1299,3.18,7173
7,2019,421,1294,3.07,8467
8,2020,324,932,2.88,9399
9,2021,472,1313,2.78,10712


In [87]:
# ─────────────────────────────────────────────────────────────────────────────
# Query 9 — SUBQUERY (inline): Keepers with above-average Save% for their year.
# WHY: Find overachieving keepers relative to the league standard.
# HOW: Inline subquery in WHERE clause computes per-year avg save%.
# ─────────────────────────────────────────────────────────────────────────────
q9 = pd.read_sql_query("""
    SELECT
        g.Player,
        g.Club,
        g.Year,
        ROUND(g."Sv%", 2)        AS save_pct,
        ROUND(avg.avg_sv, 2)     AS league_avg_sv,
        ROUND(g."Sv%" - avg.avg_sv, 2) AS above_avg_by
    FROM goalkeepers g
    JOIN (
        SELECT Year, AVG("Sv%") AS avg_sv
        FROM goalkeepers
        WHERE Season = 'reg' AND GP >= 10
        GROUP BY Year
    ) avg ON g.Year = avg.Year
    WHERE g.Season = 'reg'
      AND g.GP     >= 10
      AND g."Sv%"  > avg.avg_sv
    ORDER BY above_avg_by DESC
    LIMIT 20
""", conn)
print('Query 9 — Keepers with Above-Average Save% (Inline Subquery)')
q9

Query 9 — Keepers with Above-Average Save% (Inline Subquery)


,Player,Club,Year,save_pct,league_avg_sv,above_avg_by
0,Tyler Deric,HOU,2016,83.80,68.46,15.34
1,Bill Hamid,DC,2018,81.00,66.91,14.09
2,Brad Guzan,ATL,2017,80.90,66.90,14.00
3,Jimmy Maurer,DAL,2020,79.40,67.25,12.15
4,Jesse Gonzalez,DAL,2015,78.70,67.16,11.54
5,Tim Melia,SKC,2017,78.40,66.90,11.50
6,Andre Blake,PHI,2020,77.80,67.25,10.55
7,Steve Clark,POR,2019,77.10,66.72,10.38
8,Stefan Frei,SEA,2015,77.10,67.16,9.94
9,Bill Hamid,DC,2012,77.90,68.09,9.81


In [88]:
# ─────────────────────────────────────────────────────────────────────────────
# Query 10 — CTE (subquery style): Best career save% among keepers
#            who played ≥5 seasons and ≥100 games.
# WHY: Identify all-time elite keepers on a large enough sample.
# HOW: CTE aggregates career totals; outer query filters and ranks.
# ─────────────────────────────────────────────────────────────────────────────
q10 = pd.read_sql_query("""
    WITH career AS (
        SELECT
            Player,
            COUNT(DISTINCT Year)            AS seasons_played,
            SUM(GP)                         AS career_gp,
            SUM(SV)                         AS career_saves,
            SUM(GA)                         AS career_ga,
            SUM(ShO)                        AS career_shutouts,
            SUM(W)                          AS career_wins,
            ROUND(
                SUM(SV) * 100.0 / NULLIF(SUM(SV) + SUM(GA), 0)
            , 2)                            AS career_save_pct
        FROM goalkeepers
        WHERE Season = 'reg'
        GROUP BY Player
    )
    SELECT *
    FROM career
    WHERE seasons_played >= 5
      AND career_gp      >= 100
    ORDER BY career_save_pct DESC
    LIMIT 20
""", conn)
print('Query 10 — All-Time Best Keepers (CTE, ≥5 seasons, ≥100 GP)')
q10

Query 10 — All-Time Best Keepers (CTE, ≥5 seasons, ≥100 GP)


,Player,seasons_played,career_gp,career_saves,career_ga,career_shutouts,career_wins,career_save_pct
0,Bill Hamid,9,212,777,267,62,80,74.43
1,Stefan Frei,9,221,677,267,63,106,71.72
2,Tim Melia,9,173,530,217,52,73,70.95
3,Steve Clark,7,160,542,228,37,66,70.39
4,Nick Rimando,8,226,644,275,67,102,70.08
5,Sean Johnson,9,251,779,336,57,101,69.87
6,Brian Rowe,9,105,340,148,24,33,69.67
7,Andre Blake,7,145,460,204,38,59,69.28
8,David Bingham,9,180,598,271,46,67,68.81
9,David Ousted,7,173,514,235,46,63,68.62


In [89]:
# ─────────────────────────────────────────────────────────────────────────────
# Query 11 — GROUP BY ROLLUP simulation: Goals allowed by Club AND Season type.
# WHY: Compare regular-season vs playoff defensive performance per club.
# HOW: SQLite lacks ROLLUP; we UNION the detail and subtotal levels manually
#      to replicate GROUP BY ROLLUP(Club, Season) behavior.
# ─────────────────────────────────────────────────────────────────────────────
q11 = pd.read_sql_query("""
    -- Detail: Club × Season
    SELECT Club, Season,
           ROUND(AVG(GAA), 3) AS avg_GAA,
           SUM(GA)            AS total_GA,
           COUNT(*)           AS keeper_seasons,
           'detail'           AS rollup_level
    FROM goalkeepers
    GROUP BY Club, Season

    UNION ALL

    -- Subtotal: Club only (Season rolled up)
    SELECT Club, 'ALL',
           ROUND(AVG(GAA), 3),
           SUM(GA),
           COUNT(*),
           'club_subtotal'
    FROM goalkeepers
    GROUP BY Club

    UNION ALL

    -- Grand total
    SELECT 'ALL', 'ALL',
           ROUND(AVG(GAA), 3),
           SUM(GA),
           COUNT(*),
           'grand_total'
    FROM goalkeepers

    ORDER BY Club, Season
""", conn)
print('Query 11 — ROLLUP: Goals Allowed by Club × Season Type')
q11.head(20)

Query 11 — ROLLUP: Goals Allowed by Club × Season Type


,Club,Season,avg_GAA,total_GA,keeper_seasons,rollup_level
0,ALG,ALL,1.40,13,2,club_subtotal
1,ALG,reg,1.40,13,2,detail
2,ALL,ALL,1.00,8949,650,grand_total
3,ATL,ALL,0.89,170,18,club_subtotal
4,ATL,reg,0.89,170,18,detail
5,BRA,ALL,1.29,9,1,club_subtotal
6,BRA,reg,1.29,9,1,detail
7,CAN,ALL,0.00,0,1,club_subtotal
8,CAN,reg,0.00,0,1,detail
9,CHI,ALL,1.29,414,25,club_subtotal


In [90]:
# ─────────────────────────────────────────────────────────────────────────────
# Query 12 — Window + JOIN: Season rank of each keeper AND their club's
#            home win rate, to see if great keepers correlate with home form.
# WHY: Combine player-level and team-level info in one analytical result.
# HOW: CTE builds home win rates from matches; second CTE ranks keepers;
#      final SELECT JOINs them on Club_Full = home team name.
# ─────────────────────────────────────────────────────────────────────────────
q12 = pd.read_sql_query("""
    WITH home_rates AS (
        SELECT
            home AS club,
            year,
            COUNT(*) AS games,
            ROUND(100.0 * SUM(CASE WHEN home_score > away_score THEN 1 ELSE 0 END)
                  / COUNT(*), 1) AS home_win_pct
        FROM matches
        WHERE home_score IS NOT NULL
        GROUP BY home, year
    ),
    ranked_keepers AS (
        SELECT
            Player, Club_Full, Year,
            ROUND("Sv%", 2) AS save_pct,
            GAA,
            RANK() OVER (PARTITION BY Year ORDER BY "Sv%" DESC) AS yr_rank
        FROM goalkeepers
        WHERE Season = 'reg' AND GP >= 10
    )
    SELECT
        rk.Player,
        rk.Club_Full,
        rk.Year,
        rk.save_pct,
        rk.GAA,
        rk.yr_rank,
        hr.home_win_pct
    FROM ranked_keepers rk
    LEFT JOIN home_rates hr
        ON rk.Club_Full = hr.club
        AND rk.Year     = hr.year
    WHERE rk.yr_rank <= 5
    ORDER BY rk.Year, rk.yr_rank
""", conn)
print('Query 12 — Top-5 Keepers per Year with Club Home Win % (Window + JOIN)')
q12.head(20)

Query 12 — Top-5 Keepers per Year with Club Home Win % (Window + JOIN)


,Player,Club_Full,Year,save_pct,GAA,yr_rank,home_win_pct
0,Bill Hamid,.,2012,77.90,1.00,1,NaN
1,Michael Gspurning,Seattle Sounders,2012,76.60,0.71,2,NaN
2,Jimmy Nielsen,KC Wiz,2012,75.50,0.79,3,NaN
3,Sean Johnson,NYCFC,2012,75.50,1.23,3,NaN
4,Brad Knighton,New England,2012,75.00,0.88,5,NaN
5,Matt Reis,New England,2013,78.40,0.67,1,NaN
6,Joe Cannon,Vancouver,2013,73.80,1.40,2,NaN
7,Nick Rimando,Real Salt Lake,2013,73.20,1.04,3,63.20
8,Donovan Ricketts,Portland,2013,73.00,0.97,4,NaN
9,Jon Busch,San Jose,2013,71.10,1.24,5,NaN
